# 2.2.2 HUIM with tuned freshness

## 1. Business hypothesis and experiment overview

**Owner:** Winston Ho Rui Wei  
**Status:** completed run `20261104_234712_freshness_seed42`; executed outputs and findings are preserved below.

The initial HUIM model recovered known clicks better than the initial LSTM,
while the LSTM study found a strong freshness effect. This experiment tests
whether HUIM also benefits from a fitted publication-age penalty. It **reloads
the completed HUIM weights and fitted preprocessing without training**. The
score is `neural_logit - weight * log1p(article_age_hours)`.

Select the weight on the same fixed **2,000 validation requests** as improved
LSTM. Maximise macro known-click Recall@100, then observed-click nDCG@100, then
prefer the smaller coefficient. Evaluate the chosen recipe on the same separate
5,000-request reporting sample as the LSTM ablation study and on all validation
requests. These remain development comparisons, not an untouched final test.

Every request is scored against the complete time-eligible union catalogue.
Export the genuine top 200, retaining the first 100 as the working shortlist.
All recorded-slate candidates are scored separately. No clicked or shown articles
are inserted into the catalogue shortlist. Holdout labels remain masked.

The inference path preserves HUIM's neural forward calculation, reuses cached
states, ranks all trial coefficients from one neural pass, and replaces repeated
full sorts with exact GPU top-k. The separate business reranker is unchanged.


## 2. Setup and run configuration

Use a fresh **A100 GPU / High RAM** Colab runtime. Upload
**`huim_freshness_helpers.zip`** to `/content`, leave it zipped, then Run all and
authorise Drive. The shared folder must be available through the existing live
My Drive shortcut. No GitHub clone or credentials are needed.

The notebook uses the trained source run below. Its saved helper code, weights,
preprocessing and original feature mappings are authoritative. To reproduce this
continuation from another completed initial HUIM fit, change only `SOURCE_RUN_ID`
to that run's folder name. The initial completed notebook is preserved separately.


In [2]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import sys, subprocess, importlib.util, importlib, json, shutil, time, hashlib, sqlite3, zipfile

from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = next(p for p in [
    Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder'),
    Path('/content/drive/MyDrive/BT4222 Shared Final Folder')
] if (p/'split_manifest.json').is_file())

SOURCE_RUN_ID = '20261104_014752_seed42'
SOURCE_RUN = DATA_ROOT/'Experiments'/'2.2_huim'/SOURCE_RUN_ID
RUN_ID = datetime.now(ZoneInfo('Asia/Singapore')).strftime('%Y%m%d_%H%M%S') + '_freshness_seed42'
RUN_DIR = Path('/content/huim_freshness_runs')/RUN_ID
PERSIST_DIR = DATA_ROOT/'Experiments'/'2.2_huim'/'2.2.2_freshness_run'/RUN_ID
CODE_DIR = Path('/content/huim_freshness_code')
for p in (RUN_DIR, PERSIST_DIR, CODE_DIR):
    p.mkdir(parents=True, exist_ok=True)

missing = [p for p in ('duckdb', 'optuna') if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
with zipfile.ZipFile('/content/huim_freshness_helpers.zip') as z:
    z.extractall(CODE_DIR)
shutil.copytree(SOURCE_RUN/'model'/'code'/'huim', CODE_DIR/'huim', dirs_exist_ok=True)
sys.path.insert(0, str(CODE_DIR))

import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq, torch, optuna
from huim import pipeline as P, catalogue as C, data as D, sampling as A
from huim.persist import load_model, check_article_ids
from huim.io_utils import read_json, atomic_write_json, RunLogger, environment_info
from huim.train import score_recorded
from huim.metrics import impression_metrics
import huim_freshness as H

assert torch.cuda.is_available(), 'Select an A100 GPU runtime before running.'
DEVICE = torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
SEED, SELECTION_N, REPORT_N, EXPORT_K = 42, 2000, 5000, 200
REQUEST_BATCH, PAIR_CHUNK = 16, 131072

run_logger = RunLogger(RUN_DIR)
def log(*parts):
    run_logger(" ".join(map(str, parts)))
log.stage = run_logger.stage

log('Run:', RUN_ID, '| GPU:', torch.cuda.get_device_name(), '| source model:', SOURCE_RUN_ID)
log('Persistent folder:', PERSIST_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2026-11-04 15:47:12 Run: 20261104_234712_freshness_seed42 | GPU: NVIDIA A100-SXM4-80GB | source model: 20261104_014752_seed42
2026-11-04 15:47:12 Persistent folder: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.2_huim/2.2.2_freshness_run/20261104_234712_freshness_seed42


## 3. Reload the selected model and shared data

Use the source model's saved preprocessing without fitting new vocabularies,
bins or scalers. Build the 20,738-article SMALL lookup and fixed pre-period histories.
Article row positions are constructed together with their candidate/history
lookups; the saved preprocessing preserves categorical and numeric encodings. Keep articles missing vectors. The existing data loader masks project
holdout labels. Reading source data and reconstructing tensors is required;
neural training and the original hyperparameter search are not repeated.


In [3]:
CFG = read_json(SOURCE_RUN/'tuning'/'tuned_config.json')
model, MODEL_CONFIG, pre, MODEL_METADATA = load_model(SOURCE_RUN, DEVICE)
model.requires_grad_(False)
with log.stage('reload shared inputs with saved preprocessing'):
    ws = P.prepare_inputs(CFG, DATA_ROOT, pre=pre, log=log)
ART, CTX = P.device_and_tensors(ws, DEVICE)
ELIGIBILITY = {s: C.catalogue_eligibility(ws.ctx[s], ws.art) for s in D.SPLITS}
assert ws.pre.to_json() == read_json(SOURCE_RUN/'model'/'preprocessing.json')
log('Recorded requests:', {s: len(ws.ctx[s].requests) for s in D.SPLITS})
log('Eligible scoring pairs:', {s: int(v['eligible_count'].sum()) for s,v in ELIGIBILITY.items()})
log('Source neural checkpoint:', MODEL_METADATA['selection'])


2026-11-04 15:47:15 [start] reload shared inputs with saved preprocessing
2026-11-04 15:48:00 articles: 125,541 unique IDs from ['train_val', 'test']
2026-11-04 15:48:15 vectors: text 20,997, image 17,243
2026-11-04 15:48:54 recorded pairs: train 2,585,747, validation 344,666, test 348,951
2026-11-04 15:49:07 inputs ready in 112s
2026-11-04 15:49:07 [done] reload shared inputs with saved preprocessing (111.7s)
2026-11-04 15:49:08 Recorded requests: {'train': 232887, 'validation': 29110, 'test': 29142}
2026-11-04 15:49:08 Eligible scoring pairs: {'train': 28552935815, 'validation': 3608174696, 'test': 3611848189}
2026-11-04 15:49:08 Source neural checkpoint: {'version': 'catalogue_recall@100_v2_union_pool_hash_requests', 'objective': 'validation catalogue known-click Recall@100 (macro over requests with known clicks)', 'direction': 'maximize', 'secondary': 'observed-click nDCG@100 (ideal over all original known clicks, capped at 100)', 'tie_breaker': 'higher Recall@100, then higher nDCG

## 4. Declare selection and reporting requests

Selection uses the shared BLAKE2b seed-42 priority on validation behaviour IDs.
Reporting excludes those 2,000 requests and selects 5,000 using the original
`validation_ablation_catalogue` salt. Exact IDs and set fingerprints are saved.
Only requests are sampled; the article catalogue is never sampled. Group keys
are `(user_id, impression_id)`, represented as `behaviour_id="user:impression"`.


In [4]:
VAL_REQUESTS = ws.ctx['validation'].requests
SELECTION = A.hash_prefix(VAL_REQUESTS, 'validation', SELECTION_N, seed=SEED)
remaining = np.setdiff1d(np.arange(len(VAL_REQUESTS)), SELECTION)
bids = VAL_REQUESTS.behaviour_id.to_numpy().astype(str)
priorities = np.array([A.priority(SEED, 'validation_ablation_catalogue', b) for b in bids[remaining]])
report_positions = remaining[np.lexsort((bids[remaining], priorities))[:REPORT_N]]
REPORT = A.chronological(VAL_REQUESTS, report_positions)
assert len(np.intersect1d(SELECTION, REPORT)) == 0
DECLARATION = {
    'objective': 'known_click_recall@100', 'tiebreak': 'observed_click_ndcg@100, then smaller weight',
    'selection_k': 100, 'export_k': 200, 'seed': SEED, 'freshness_grid': list(H.FRESHNESS_GRID),
    'selection_requests': len(SELECTION), 'reporting_requests': len(REPORT),
    'selection_fingerprint': A.ids_digest(bids[SELECTION]),
    'reporting_fingerprint': A.ids_digest(bids[REPORT]),
    'selection_salt': 'validation', 'reporting_salt': 'validation_ablation_catalogue',
    'reporting_excludes_selection': True,
    'holdout': 'outcomes masked; final evaluation is deferred until full pipeline freeze',
}
(RUN_DIR/'selection').mkdir()
for name, positions in [('validation_selection', SELECTION), ('validation_reporting', REPORT)]:
    VAL_REQUESTS.iloc[positions][['behaviour_id','user_id','impression_id']].to_parquet(
        RUN_DIR/'selection'/f'{name}_requests.parquet', index=False)
atomic_write_json(RUN_DIR/'selection'/'declaration.json', DECLARATION)
display(pd.DataFrame([{'sample': k, 'requests': len(v)} for k,v in [('selection',SELECTION),('reporting',REPORT)]]))


,sample,requests
0,selection,2000
1,reporting,5000


## 5. Benchmark inference and select the freshness coefficient

First check the new ranker against the saved model's original neural forward
path on 16 selection requests, including zero-weight catalogue ordering.
Ranking uses exact float32 score order, then original article ID, with no epsilon.

Then evaluate all nine weights from the same neural logits for each selection
block. Optuna records the fixed grid results; these are inference evaluations
of one frozen model, not nine neural fits. The selected recipe is persisted
before the long export. No weight is selected from reporting or holdout results.


In [5]:
scorer = H.FreshnessScorer(model, CTX['validation'], ART, DEVICE, PAIR_CHUNK)
bench_pos = SELECTION[:16]
reference = C.CatalogueScorer(model, CTX['validation'], ART, DEVICE, pair_chunk=PAIR_CHUNK, amp=False)
torch.cuda.synchronize(); t0 = time.perf_counter()
old_top = reference.topk(bench_pos, 200)
torch.cuda.synchronize(); old_seconds = time.perf_counter()-t0
torch.cuda.synchronize(); t0 = time.perf_counter()
new_top = scorer.rank_many(bench_pos, [0.], 200)[0.]
torch.cuda.synchronize(); new_seconds = time.perf_counter()-t0
old_ids = ART.ids[CTX['validation'].catalogue_idx][old_top['positions'].cpu().numpy()]
lengths = new_top['lengths']
mask = np.arange(new_top['article_ids'].shape[1])[None,:] < lengths[:,None]
old_scores = old_top['scores'].cpu().numpy()
BENCHMARK = {
    'requests': len(bench_pos), 'original_seconds': old_seconds, 'new_seconds': new_seconds,
    'top200_speedup': old_seconds/new_seconds,
    'max_abs_score_difference': float(np.max(np.abs(old_scores[mask]-new_top['scores'][mask]), initial=0)),
    'rankings_match': bool(np.array_equal(old_ids[mask], new_top['article_ids'][mask])),
    'precision': 'float32; TF32 off; identical neural forward implementation',
}
assert BENCHMARK['rankings_match'] and BENCHMARK['max_abs_score_difference'] <= 1e-4
atomic_write_json(RUN_DIR/'inference_benchmark.json', BENCHMARK)
display(pd.Series(BENCHMARK))
del reference, old_top


,0
requests,16
original_seconds,0.953659
new_seconds,0.399713
top200_speedup,2.385859
max_abs_score_difference,0.0
rankings_match,True
precision,float32; TF32 off; identical neural forward im...


In [6]:
selection_rows = {w: [] for w in H.FRESHNESS_GRID}
with log.stage('freshness grid on 2000 selection requests'):
    for start in range(0, len(SELECTION), REQUEST_BATCH):
        positions = SELECTION[start:start+REQUEST_BATCH]
        ranked = scorer.rank_many(positions, H.FRESHNESS_GRID, k=100)
        for weight, result in ranked.items():
            _, rows = H.retrieval(VAL_REQUESTS, positions, result['article_ids'], result['lengths'])
            selection_rows[weight].append(rows)
        if start % (REQUEST_BATCH*25) == 0:
            log('Selection:', min(start+REQUEST_BATCH,len(SELECTION)), '/',len(SELECTION))
selection_metrics = {w: H.summarise_retrieval(pd.concat(parts, ignore_index=True))
                     for w, parts in selection_rows.items()}
(RUN_DIR/'tuning').mkdir()
study_path = RUN_DIR/'tuning'/'freshness_study.sqlite3'
study = optuna.create_study(direction='maximize', study_name=f'huim_freshness_{RUN_ID}',
    storage=f'sqlite:///{study_path}',
    sampler=optuna.samplers.GridSampler({'freshness_weight': list(H.FRESHNESS_GRID)}, seed=SEED))
def objective(trial):
    weight = trial.suggest_categorical('freshness_weight', list(H.FRESHNESS_GRID))
    metrics = selection_metrics[float(weight)]
    trial.set_user_attr('observed_click_ndcg@100', metrics['observed_click_ndcg@100'])
    trial.set_user_attr('selection_requests', len(SELECTION))
    return metrics['known_click_recall@100']
study.optimize(objective, n_trials=len(H.FRESHNESS_GRID))
WEIGHT = max(H.FRESHNESS_GRID, key=lambda w: (selection_metrics[w]['known_click_recall@100'],
                  selection_metrics[w]['observed_click_ndcg@100'], -w))
study.trials_dataframe().to_csv(RUN_DIR/'tuning'/'trials.csv', index=False)
summary = pd.DataFrame([{'freshness_weight':w, **selection_metrics[w]} for w in H.FRESHNESS_GRID])
summary.to_csv(RUN_DIR/'tuning'/'freshness_grid.csv', index=False)
atomic_write_json(RUN_DIR/'tuning'/'selection.json', {'selected_weight':WEIGHT,
    'selected_metrics':selection_metrics[WEIGHT], 'neural_fits':0, 'grid_evaluations':len(H.FRESHNESS_GRID)})
display(summary[['freshness_weight','known_click_recall@100','observed_click_ndcg@100']])
log('Selected coefficient:', WEIGHT)
del selection_rows


2026-11-04 15:49:10 [start] freshness grid on 2000 selection requests
2026-11-04 15:49:10 Selection: 16 / 2000
2026-11-04 15:49:20 Selection: 416 / 2000
2026-11-04 15:49:29 Selection: 816 / 2000
2026-11-04 15:49:38 Selection: 1216 / 2000
2026-11-04 15:49:47 Selection: 1616 / 2000
2026-11-04 15:49:56 [done] freshness grid on 2000 selection requests (46.5s)


[I 2026-11-04 15:49:58,640] A new study created in RDB with name: huim_freshness_20261104_234712_freshness_seed42
[I 2026-11-04 15:49:58,731] Trial 0 finished with value: 0.8834 and parameters: {'freshness_weight': 16.0}. Best is trial 0 with value: 0.8834.
[I 2026-11-04 15:49:58,796] Trial 1 finished with value: 0.8311499999999999 and parameters: {'freshness_weight': 0.25}. Best is trial 0 with value: 0.8834.
[I 2026-11-04 15:49:58,863] Trial 2 finished with value: 0.8859 and parameters: {'freshness_weight': 4.0}. Best is trial 2 with value: 0.8859.
[I 2026-11-04 15:49:58,928] Trial 3 finished with value: 0.742525 and parameters: {'freshness_weight': 0.0}. Best is trial 2 with value: 0.8859.
[I 2026-11-04 15:49:58,993] Trial 4 finished with value: 0.8804 and parameters: {'freshness_weight': 64.0}. Best is trial 2 with value: 0.8859.
[I 2026-11-04 15:49:59,060] Trial 5 finished with value: 0.8623999999999999 and parameters: {'freshness_weight': 0.5}. Best is trial 2 with value: 0.8859.

,freshness_weight,known_click_recall@100,observed_click_ndcg@100
0,0.00,0.742525,0.210440
1,0.25,0.831150,0.250330
2,0.50,0.862400,0.270748
3,1.00,0.879900,0.287077
4,2.00,0.887900,0.274505
5,4.00,0.885900,0.258340
6,8.00,0.886400,0.247837
7,16.00,0.883400,0.242383
8,64.00,0.880400,0.237388


2026-11-04 15:49:59 Selected coefficient: 2.0


## 6. Validation reporting sample

Compare the selected blend, the same frozen neural model at weight zero, and
newest-first on the separate 5,000 requests. If the selected weight is zero,
the first two rows are intentionally identical. This sample matches the LSTM
ablation reporting sample; it does not replace the all-validation handoff.
The selection rule is frozen before inspecting these results.


In [7]:
report_parts = {'selected_blend':[], 'raw_neural':[], 'newest_first':[]}
with log.stage('separate 5000-request reporting comparison'):
    for start in range(0, len(REPORT), REQUEST_BATCH):
        positions = REPORT[start:start+REQUEST_BATCH]
        ranked = scorer.rank_many(positions, sorted(set([0.,WEIGHT])), k=100)
        for name, weight in [('selected_blend',WEIGHT),('raw_neural',0.)]:
            result = ranked[weight]
            _, rows = H.retrieval(VAL_REQUESTS, positions, result['article_ids'], result['lengths'])
            report_parts[name].append(rows)
        ids, lengths = scorer.newest(positions,100)
        _, rows = H.retrieval(VAL_REQUESTS, positions, ids, lengths)
        report_parts['newest_first'].append(rows)
REPORT_METRICS = {}
(RUN_DIR/'evaluation').mkdir()
for name, parts in report_parts.items():
    frame = pd.concat(parts,ignore_index=True)
    frame.to_parquet(RUN_DIR/'evaluation'/f'{name}_5000_requests.parquet', index=False)
    REPORT_METRICS[name] = H.summarise_retrieval(frame)
display(pd.DataFrame(REPORT_METRICS).T)
del report_parts, scorer
torch.cuda.empty_cache()


2026-11-04 15:49:59 [start] separate 5000-request reporting comparison
2026-11-04 15:51:49 [done] separate 5000-request reporting comparison (109.8s)


,known_click_recall@100,observed_click_ndcg@100,requests_evaluated,requests_without_known_clicks,clicked_articles_total
selected_blend,0.894133,0.272563,5000.0,0.0,5021.0
raw_neural,0.753900,0.210549,5000.0,0.0,5021.0
newest_first,0.887067,0.235612,5000.0,0.0,5021.0


## 7. Persist the score recipe and export all requests

Save the original selected weights, source code, named feature mappings and
preprocessing together with the **new score recipe**. The weights are unchanged;
the recipe is necessary to reconstruct the final ranking score. Save the trial
history and SQLite study snapshot as well.

Export order is **validation → holdout → train**, so the evaluator's complete
29,110-request validation handoff reaches Drive first. Every split retains top
200 from the full eligible catalogue, plus separate complete recorded-slate
scores. Generic `score` is the blended ranking value; a sigmoid of that value is
not exported as a click probability. The working relevance cutoff stays 100.

Finished Parquets are copied once from local runtime storage to shared Drive.
Per-split manifests check complete request coverage and eligible counts without
another catalogue scoring pass. Train predictions are in-sample; holdout outcome
metrics are not computed.


In [8]:
SCORE_RECIPE = {
    'model_family':'HUIM-inspired attention with tuned freshness',
    'source_neural_run':SOURCE_RUN_ID, 'freshness_weight':float(WEIGHT),
    'formula':'neural_logit - freshness_weight * log1p(age_hours)',
    'age_hours':'max(0, request_time - published_time) / 3600; shared data has whole-second timestamps',
    'missing_publication':'excluded from catalogue; zero age penalty for separate recorded-slate diagnostics',
    'score_semantics':'generic ranking score; not calibrated click probability',
    'selection':DECLARATION, 'selected_metrics':selection_metrics[WEIGHT],
}
shutil.copytree(SOURCE_RUN/'model', RUN_DIR/'model', dirs_exist_ok=True)
# Persist the article rows used by this scoring pipeline.
np.save(RUN_DIR/'model'/'article_ids.npy', ws.art.ids)
shutil.copy2(CODE_DIR/'huim_freshness.py', RUN_DIR/'model'/'code'/'huim_freshness.py')
atomic_write_json(RUN_DIR/'model'/'score_recipe.json', SCORE_RECIPE)
shutil.copy2(SOURCE_RUN/'training_history.json', RUN_DIR/'source_training_history.json')
shutil.copy2(SOURCE_RUN/'tuning'/'summary.json', RUN_DIR/'source_neural_tuning_summary.json')
shutil.copy2(SOURCE_RUN/'tuning'/'tuned_config.json', RUN_DIR/'source_neural_config.json')
shutil.copy2(DATA_ROOT/'split_manifest.json', RUN_DIR/'split_manifest.json')
atomic_write_json(RUN_DIR/'input_manifest.json', D.input_manifest(ws.paths))
atomic_write_json(RUN_DIR/'environment.json', environment_info())

RUN_CONFIG = {
    'experiment_id':'2.2_huim', 'experiment_stage':'2.2.2_freshness_run', 'owner':'Winston Ho Rui Wei',
    'run_id':RUN_ID, 'source_neural_run':SOURCE_RUN_ID, 'neural_training_performed':False,
    'dataset_split':read_json(DATA_ROOT/'split_manifest.json'),
    'pool_policy':ws.reports['pool_policy'], 'pool_fingerprint':ws.reports['pool_fingerprint'],
    'catalogue_sources':['train_val/articles.parquet','test/articles.parquet'],
    'prediction_pool':'time_eligible_catalogue', 'request_grouping':['user_id','impression_id'],
    'selection_k':100, 'working_shortlist_size':100, 'export_k':200, 'shortlist_size':200,
    'final_display_size':5, 'freshness_weight':float(WEIGHT), 'selection':DECLARATION,
    'score_recipe':'model/score_recipe.json', 'score_semantics':SCORE_RECIPE['score_semantics'],
    'export_schema_version':5, 'inference':{'precision':'float32','pair_chunk':PAIR_CHUNK,
                                         'request_batch':REQUEST_BATCH,'benchmark':BENCHMARK},
    'export_request_scope':'all requests in all splits', 'train_predictions':'in-sample; not out-of-fold',
    'holdout_labels':'masked; no outcome evaluation', 'catalogue_exports':{}, 'recorded_slate_exports':{},
    'source_weights_sha256':MODEL_METADATA['weights_sha256'],
    'model':MODEL_CONFIG.to_json(), 'source_neural_checkpoint':MODEL_METADATA['selection'],
    'preprocessing':'model/preprocessing.json', 'feature_dictionary':'model/feature_dictionary.json',
    'persistent_artifact_location':str(PERSIST_DIR),
}
METRICS = {'partition':'validation development; selection requests reused for tuning',
           'selection':selection_metrics[WEIGHT], 'reporting_5000':REPORT_METRICS,
           'full_validation_catalogue':None, 'holdout':{'metrics':None,'evaluation':'deferred'}}

def persist_file(path):
    path = Path(path)
    target = PERSIST_DIR/path.relative_to(RUN_DIR)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path,target)

def persist_metadata():
    atomic_write_json(RUN_DIR/'run_config.json',RUN_CONFIG)
    atomic_write_json(RUN_DIR/'metrics.json',METRICS)
    for file in RUN_DIR.glob('*.json'):
        persist_file(file)

# A consistent SQLite backup can be copied while the study's engine remains open.
with sqlite3.connect(study_path) as db, sqlite3.connect(PERSIST_DIR/'freshness_study.sqlite3') as backup:
    db.backup(backup)
for folder in ['model','selection','tuning','evaluation']:
    for file in (RUN_DIR/folder).rglob('*'):
        if file.is_file() and file != study_path:
            persist_file(file)
for file in [RUN_DIR/'source_training_history.json',RUN_DIR/'source_neural_tuning_summary.json',
             RUN_DIR/'source_neural_config.json']:
    persist_file(file)
persist_metadata()
log('Selected model and freshness recipe safely saved:', PERSIST_DIR)


2026-11-04 15:52:04 Selected model and freshness recipe safely saved: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.2_huim/2.2.2_freshness_run/20261104_234712_freshness_seed42


In [9]:
def export_recorded_split(split):
    with log.stage(f'{split}: recorded-slate scores using the selected recipe'):
        ctx = CTX[split]
        with torch.inference_mode():
            raw = score_recorded(model,ctx,ART,DEVICE,amp=False)
        age = H.recorded_age(ctx,ART)
        path = RUN_DIR/'diagnostics'/'recorded_slates'/f'{split}_scores.parquet'
        final = H.write_recorded(path,ctx,raw,age,WEIGHT)
        info = {'path':str(path.relative_to(RUN_DIR)), 'rows':len(final), 'score_semantics':SCORE_RECIPE['score_semantics']}
        if split == 'validation':
            METRICS['full_validation_recorded_slates'] = impression_metrics(
                ctx.recorded['req_pos'],ctx.recorded['label'].astype(int),final,k=5)
            METRICS['full_validation_raw_neural_recorded_slates'] = impression_metrics(
                ctx.recorded['req_pos'],ctx.recorded['label'].astype(int),raw,k=5)
        persist_file(path)
        RUN_CONFIG['recorded_slate_exports'][split] = info
        log(split,'recorded-slate rows:',len(final))
        reference = {'raw':raw[:128].copy(), 'final':final[:128].copy()} if split == 'validation' else None
        persist_metadata()
        return reference


In [10]:
EXPORTS = {}
with log.stage('all-request top-200 exports'):
    for split in ['validation','test','train']:
        reference = export_recorded_split(split)
        if split == 'validation':
            RECORDED_REFERENCE = reference
        ctx = CTX[split]
        requests = ws.ctx[split].requests
        scorer = H.FreshnessScorer(model,ctx,ART,DEVICE,PAIR_CHUNK)
        weights = sorted(set([0.,WEIGHT])) if split == 'validation' else [WEIGHT]
        writer = H.ShortlistWriter(RUN_DIR,split)
        metric_parts = {w:[] for w in weights}
        newest_parts = []
        t0, next_log = time.time(), 1000
        for start in range(0,len(requests),REQUEST_BATCH):
            positions = np.arange(start,min(start+REQUEST_BATCH,len(requests)))
            ranked = scorer.rank_many(positions,weights,200)
            result = ranked[WEIGHT]
            assert np.array_equal(result['eligible_count'],ELIGIBILITY[split]['eligible_count'][positions])
            writer.add(requests.iloc[positions],result)
            if split == 'validation':
                for weight, r in ranked.items():
                    _, rows = H.retrieval(requests,positions,r['article_ids'],r['lengths'])
                    metric_parts[weight].append(rows)
                ids, lengths = scorer.newest(positions,100)
                _, rows = H.retrieval(requests,positions,ids,lengths)
                newest_parts.append(rows)
            done = int(positions[-1])+1
            if done >= next_log:
                elapsed = time.time()-t0
                log(split,done,'/',len(requests),'requests | elapsed min',round(elapsed/60,1),
                    '| ETA min',round(elapsed/done*(len(requests)-done)/60,1))
                next_log += 1000
        files = writer.close()
        assert writer.n_requests == len(requests)
        assert writer.eligible_pairs == int(ELIGIBILITY[split]['eligible_count'].sum())
        assert writer.n_scores == int(np.minimum(200,ELIGIBILITY[split]['eligible_count']).sum())
        # One compressed ranking row per request; check IDs/order after persistence as well.
        ranking_table = pq.read_table(files['rankings'], columns=['behaviour_id'])
        assert ranking_table['behaviour_id'].to_pylist() == requests.behaviour_id.astype(str).tolist()
        assert pq.ParquetFile(files['scores']).metadata.num_rows == writer.n_scores
        info = {'requests':writer.n_requests,'eligible_pairs':writer.eligible_pairs,
                'retained_score_rows':writer.n_scores,'seconds':time.time()-t0,
                'empty_pools':int((ELIGIBILITY[split]['eligible_count']==0).sum()),
                'request_coverage_and_counts_checked':True,
                'files':{k:str(v.relative_to(RUN_DIR)) for k,v in files.items()}}
        if split == 'validation':
            METRICS['full_validation_catalogue'] = H.summarise_retrieval(pd.concat(metric_parts[WEIGHT],ignore_index=True))
            METRICS['full_validation_raw_neural_catalogue'] = H.summarise_retrieval(pd.concat(metric_parts[0.],ignore_index=True))
            METRICS['full_validation_newest_first'] = H.summarise_retrieval(pd.concat(newest_parts,ignore_index=True))
            display(pd.DataFrame({k:METRICS[k] for k in ['full_validation_catalogue',
                'full_validation_raw_neural_catalogue','full_validation_newest_first']}).T)
        for file in files.values():
            persist_file(file)
        EXPORTS[split] = info
        RUN_CONFIG['catalogue_exports'][split] = info
        atomic_write_json(RUN_DIR/'export_manifest.json',{'export_schema_version':5,
            'score_recipe':SCORE_RECIPE,'selection_k':100,'export_k':200,'splits':EXPORTS})
        persist_metadata()
        log(split,'complete:',info)
        del scorer, metric_parts, newest_parts, ranking_table
        torch.cuda.empty_cache()


2026-11-04 15:52:04 [start] all-request top-200 exports
2026-11-04 15:52:04 [start] validation: recorded-slate scores using the selected recipe
2026-11-04 15:52:05 validation recorded-slate rows: 344666
2026-11-04 15:52:05 [done] validation: recorded-slate scores using the selected recipe (1.2s)
2026-11-04 15:52:27 validation 1008 / 29110 requests | elapsed min 0.4 | ETA min 10.4
2026-11-04 15:52:49 validation 2000 / 29110 requests | elapsed min 0.7 | ETA min 10.0
2026-11-04 15:53:12 validation 3008 / 29110 requests | elapsed min 1.1 | ETA min 9.6
2026-11-04 15:53:34 validation 4000 / 29110 requests | elapsed min 1.5 | ETA min 9.3
2026-11-04 15:53:56 validation 5008 / 29110 requests | elapsed min 1.8 | ETA min 8.9
2026-11-04 15:54:18 validation 6000 / 29110 requests | elapsed min 2.2 | ETA min 8.5
2026-11-04 15:54:40 validation 7008 / 29110 requests | elapsed min 2.6 | ETA min 8.2
2026-11-04 15:55:02 validation 8000 / 29110 requests | elapsed min 3.0 | ETA min 7.8
2026-11-04 15:55:24 v

,known_click_recall@100,observed_click_ndcg@100,requests_evaluated,requests_without_known_clicks,clicked_articles_total
full_validation_catalogue,0.895492,0.273506,29110.0,0.0,29249.0
full_validation_raw_neural_catalogue,0.752058,0.211205,29110.0,0.0,29249.0
full_validation_newest_first,0.890207,0.236519,29110.0,0.0,29249.0


2026-11-04 16:02:50 validation complete: {'requests': 29110, 'eligible_pairs': 3608174696, 'retained_score_rows': 5822000, 'seconds': 644.3807699680328, 'empty_pools': 0, 'request_coverage_and_counts_checked': True, 'files': {'scores': 'predictions/validation/submission_article_scores.parquet', 'rankings': 'rankings/validation/submission_top_200.parquet', 'manifest': 'diagnostics/catalogue/validation_requests.parquet'}}
2026-11-04 16:02:50 [start] test: recorded-slate scores using the selected recipe
2026-11-04 16:02:50 test recorded-slate rows: 348951
2026-11-04 16:02:50 [done] test: recorded-slate scores using the selected recipe (0.3s)
2026-11-04 16:03:12 test 1008 / 29142 requests | elapsed min 0.4 | ETA min 10.1
2026-11-04 16:03:33 test 2000 / 29142 requests | elapsed min 0.7 | ETA min 9.7
2026-11-04 16:03:55 test 3008 / 29142 requests | elapsed min 1.1 | ETA min 9.4
2026-11-04 16:04:16 test 4000 / 29142 requests | elapsed min 1.4 | ETA min 9.0
2026-11-04 16:04:38 test 5008 / 2914

## 8. Reload check from the persistent copy

Import the helper code from the new saved bundle, reload its original neural
weights and fitted preprocessing, and read its selected freshness recipe. Do not
refit. Verify named features and article-ID order, then compare raw/blended scores
for a small recorded batch and the retained catalogue ranking against the actual
saved validation exports. No catalogue-wide second pass is needed.


In [11]:
SAVED_CODE = PERSIST_DIR/'model'/'code'
for name in list(sys.modules):
    if name == 'huim' or name.startswith('huim.'):
        del sys.modules[name]
sys.path.insert(0,str(SAVED_CODE))
from huim.persist import load_model as reload_model, check_article_ids as reload_article_ids
restored, restored_cfg, restored_pre, restored_meta = reload_model(PERSIST_DIR,DEVICE)
saved_recipe = read_json(PERSIST_DIR/'model'/'score_recipe.json')
spec = importlib.util.spec_from_file_location('persisted_huim_freshness',SAVED_CODE/'huim_freshness.py')
RH = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = RH
spec.loader.exec_module(RH)
assert restored_pre.to_json() == ws.pre.to_json()
assert reload_article_ids(PERSIST_DIR,ws.art)
assert read_json(PERSIST_DIR/'model'/'feature_dictionary.json') == read_json(RUN_DIR/'model'/'feature_dictionary.json')
ctx = CTX['validation']
rec = ctx.recorded
positions = torch.as_tensor(rec['req_pos'][:128],device=DEVICE)
articles = torch.as_tensor(rec['art_idx'][:128],device=DEVICE)
with torch.inference_mode():
    raw = restored(ctx.tensors,ART.tensors,len(ART.ids),positions,articles).float().cpu().numpy()
final = RH.blend(torch.as_tensor(raw),torch.as_tensor(H.recorded_age(ctx,ART)[:len(raw)]),
                 saved_recipe['freshness_weight']).numpy()
scorer = RH.FreshnessScorer(restored,ctx,ART,DEVICE,PAIR_CHUNK)
checked = scorer.rank_many(np.arange(REQUEST_BATCH),[saved_recipe['freshness_weight']],200)[saved_recipe['freshness_weight']]
rank_path = PERSIST_DIR/'rankings'/'validation'/'submission_top_200.parquet'
saved_rankings = next(pq.ParquetFile(rank_path).iter_batches(batch_size=8)).to_pydict()
score_path = PERSIST_DIR/'predictions'/'validation'/'submission_article_scores.parquet'
saved_scores = next(pq.ParquetFile(score_path).iter_batches(batch_size=1600)).to_pydict()
expected_ids = [checked['article_ids'][j,:int(n)].tolist() for j,n in enumerate(checked['lengths'][:8])]
assert saved_rankings['behaviour_id'] == ws.ctx['validation'].requests.behaviour_id.iloc[:8].tolist()
max_raw = float(np.max(np.abs(raw-RECORDED_REFERENCE['raw'][:len(raw)]),initial=0))
max_final = float(np.max(np.abs(final-RECORDED_REFERENCE['final'][:len(raw)]),initial=0))
flat_scores = np.concatenate([checked['scores'][j,:int(n)] for j,n in enumerate(checked['lengths'][:8])])
max_catalogue = float(np.max(np.abs(flat_scores-np.asarray(saved_scores['score'][:len(flat_scores)])),initial=0))
RELOAD = {'loaded_from':str(PERSIST_DIR),'freshness_weight_reloaded':saved_recipe['freshness_weight'],
    'recorded_pairs_compared':len(raw),'catalogue_requests_compared':8,
    'max_abs_raw_difference':max_raw,'max_abs_blended_difference':max_final,
    'max_abs_catalogue_difference':max_catalogue,
    'catalogue_rankings_match':saved_rankings['article_ids']==expected_ids,
    'fitted_preprocessing_matches':True,'feature_names_match':True,'article_ids_match':True,'tolerance':1e-4}
RELOAD['passed'] = bool(max(max_raw,max_final,max_catalogue)<=1e-4 and RELOAD['catalogue_rankings_match'])
atomic_write_json(RUN_DIR/'reload_check.json',RELOAD)
persist_file(RUN_DIR/'reload_check.json')
assert RELOAD['passed'], 'Inspect reload_check.json before using this handoff.'
display(pd.Series(RELOAD))


,0
loaded_from,/content/drive/MyDrive/Colab Notebooks/BT4222 ...
freshness_weight_reloaded,2.0
recorded_pairs_compared,128
catalogue_requests_compared,8
max_abs_raw_difference,0.000001
max_abs_blended_difference,0.000002
max_abs_catalogue_difference,0.0
catalogue_rankings_match,True
fitted_preprocessing_matches,True
feature_names_match,True


## 9. Business interpretation and evaluator handoff

### Completed run and selection decision

Run `20261104_234712_freshness_seed42` reused the selected HUIM weights from
`20261104_014752_seed42` and its fitted preprocessing. **No neural model was
retrained.** The nine freshness coefficients were evaluated from the same neural
logits for each block of the fixed 2,000 validation-selection requests.

The selected coefficient was **2**, giving the final ranking score
`neural_logit - 2 * log1p(article_age_hours)`. Selection Recall@100 was 0.8879
and observed-click nDCG@100 was 0.2745. Weight 1 produced higher nDCG@100 (0.2871)
but lower Recall@100 (0.8799); choosing weight 2 followed the declared primary
objective of recovering known clicks in the working shortlist.

### Retrieval findings

All rows below use **the same 29,110 validation requests and time-eligible
catalogue**. The LSTM comparison comes from its completed improved run.

| Ranking method | Macro known-click Recall@100 | Observed-click nDCG@100 |
| --- | ---: | ---: |
| Same frozen HUIM, raw neural score | 0.7521 | 0.2112 |
| Newest-first | 0.8902 | 0.2365 |
| HUIM + selected freshness | **0.8955** | **0.2735** |
| Improved LSTM + selected freshness | 0.8970 | 0.2752 |

HUIM with freshness recovered an average **89.55% of each request's known clicks**,
compared with 75.21% for its raw score. Its gain over newest-first was **0.53
percentage point** in Recall@100 and 0.0370 in nDCG@100. On the separate 5,000
reporting requests, the same blend recovered 89.41%, compared with 75.39% for raw
HUIM and 88.71% for newest-first, giving the same ordering among these methods.

The improved LSTM blend remained slightly ahead, by **0.15 percentage point** in
full-validation Recall@100. Raw HUIM's stronger result than the initial raw LSTM
therefore did not imply a stronger freshness blend. These findings support
testing the combined recipe directly. The small difference between the two
blends has no paired uncertainty interval here and does not establish a robust
architectural advantage.

Freshness supplies most of the catalogue recovery; the neural score adds a
smaller improvement over newest-first. This retains evidence of reader interest
for the separate business reranker, but does not demonstrate higher online CTR,
reader retention or profit. Only exposed articles have observed click outcomes.
Unshown outcomes remain unknown, and the final score is not a calibrated click
probability. Validation informed development and selection; it is not an
untouched final test set.

### Efficiency and completed handoff

On the A100-SXM4-80GB, the 16-request benchmark reproduced the reference scores
exactly and matched its rankings, while top-200 inference was **2.39 times as
fast**. This isolates the new ranking path on the same GPU. Selection scoring
took 46.5 seconds and the 5,000-request reporting comparison took 109.8 seconds.

All-request catalogue export finished in **6,279.3 seconds (1 hour 45 minutes)**:
approximately 10.7 minutes for validation, 10.4 minutes for holdout and 83.4
minutes for train, with remaining copy/logging overhead. The initial HUIM export
took 7 hours 26 minutes; that larger end-to-end change includes both hardware
and implementation differences and should not be attributed solely to the
ranking optimisation.

The persistent handoff covers **29,110 validation, 29,142 holdout and 232,887
training requests**. Every request has a genuine top-200 ranking, retained-pair
scores and separate complete recorded-slate diagnostics. The first 100 remain
the working shortlist; the additional 100 are a buffer. Retained score-row counts
are 5,822,000, 5,828,400 and 46,577,400 respectively. No request had an empty
eligible pool. Training predictions are in-sample and holdout outcomes were not
evaluated.

Reload from the persistent bundle passed on **128 recorded pairs and eight
catalogue requests**. Raw/blended recorded-score differences were approximately
`1e-6`/`2e-6`, below the `1e-4` tolerance; retained catalogue scores reproduced
exactly and rankings matched. The saved coefficient was recovered as 2, and
feature names, fitted preprocessing and original article-ID order matched.

The evaluator can use `rankings/validation/submission_top_200.parquet`, retained
scores, recorded-slate scores and `run_config.json` immediately. The complete
bundle also preserves the frozen weights, source code and `model/score_recipe.json`
for later explainability without retraining. Explanations should separate the
learned neural contribution from the explicit age penalty. Final holdout
evaluation remains deferred until the full pipeline is frozen.


In [12]:
log('Handoff folder:',PERSIST_DIR)
log('Source neural run:',SOURCE_RUN_ID,'| neural fits performed: 0 | selected freshness:',WEIGHT)
log('Selection sample:',selection_metrics[WEIGHT])
log('Reporting sample:',REPORT_METRICS)
log('Full validation:',METRICS['full_validation_catalogue'])
log('All-request export coverage:',{s:v['requests'] for s,v in EXPORTS.items()})
log('Persistent numerical reload passed:',RELOAD['passed'])


2026-11-04 17:36:44 Handoff folder: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.2_huim/2.2.2_freshness_run/20261104_234712_freshness_seed42
2026-11-04 17:36:44 Source neural run: 20261104_014752_seed42 | neural fits performed: 0 | selected freshness: 2.0
2026-11-04 17:36:44 Selection sample: {'known_click_recall@100': 0.8879, 'observed_click_ndcg@100': 0.2745046750086024, 'requests_evaluated': 2000, 'requests_without_known_clicks': 0, 'clicked_articles_total': 2013}
2026-11-04 17:36:44 Reporting sample: {'selected_blend': {'known_click_recall@100': 0.8941333333333332, 'observed_click_ndcg@100': 0.27256332767908237, 'requests_evaluated': 5000, 'requests_without_known_clicks': 0, 'clicked_articles_total': 5021}, 'raw_neural': {'known_click_recall@100': 0.7539, 'observed_click_ndcg@100': 0.21054915940865243, 'requests_evaluated': 5000, 'requests_without_known_clicks': 0, 'clicked_articles_total': 5021}, 'newest_first': {'known_click_recall@100': 0.88706